In [ ]:
import os, time, datetime, random, collections
from types import SimpleNamespace as _NS
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as data
from torch.cuda import amp
from torch.utils.tensorboard import SummaryWriter
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torchtoolbox.transform import Cutout
from spikingjelly.clock_driven import functional
from spikingjelly.clock_driven import surrogate as surrogate_sj
from models import spiking_resnet_imagenet, spiking_resnet, spiking_vgg_bn
from modules import neuron
from modules import surrogate as surrogate_self
from utils import AverageMeter, accuracy
from utils.cifar10_dvs import CIFAR10DVS
from spikingjelly.datasets.dvs128_gesture import DVS128Gesture
from tqdm import tqdm
from py3nvml.py3nvml import *
import threading

Cfg = _NS(
    seed            = 2025,
    name            = '',               # 
    T               = 6,                # 
    tau             = 1.1,              # 
    b               = 1,              # batch size
    epochs          = 100,               #
    j               = 0,                # num_workers
    data_dir        = './data',
    dataset         = 'cifar100',        # cifar10 / cifar100 / DVSCIFAR10 / dvsgesture / imagenet
    out_dir         = './logs',
    surrogate       = 'triangle',       # sigmoid / rectangle / triangle
    resume          = None,             # 'path/to/checkpoint.pth'
    pre_train       = None,             # 'path/to/pretrain.pth'
    amp             = False,             
    opt             = 'SGD',            # 'SGD' 'AdamW'
    lr              = 0.00078125,
    momentum        = 0.9,
    lr_scheduler    = 'CosALR',         # 'StepLR' 'CosALR'
    step_size       = 100,
    gamma           = 0.1,
    T_max           = 300,
    model           = 'spiking_vgg11_lttt_sw',
    drop_rate       = 0.0,
    weight_decay    = 0.0,
    loss_lambda     = 0.1,             # CE + MSE
    mse_n_reg       = False,            # 
    loss_means      = 1.0,              #
    save_init       = False,
    online_update   = False,             # 
    BN              = False             #
)

random.seed(Cfg.seed)
np.random.seed(Cfg.seed)
torch.manual_seed(Cfg.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(Cfg.seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Running on:', device)

def _init_txt_logger(filename="SpikON_cifar100_vgg11.txt"):
    path = filename
    if not os.path.exists(path):
        with open(path, "w", encoding="utf-8") as f:
            f.write("epoch\ttrain_loss\ttrain_acc\ttest_loss\ttest_acc\tmax_test_acc\ttraining_epoch_latency(s)\tavg_power(W)\tenergy(J)\ttotal_time(s)\n")
    return path

def _append_txt_log(path, **kw):
    line = "{epoch}\t{train_loss:.6f}\t{train_acc:.6f}\t{test_loss:.6f}\t{test_acc:.6f}\t{max_test_acc:.6f}\t{epoch_latency:.3f}\t{avg_power:.3f}\t{energy:.3f}\t{total_time:.3f}\n".format(**kw)
    with open(path, "a", encoding="utf-8") as f:
        f.write(line)

txt_log_path = _init_txt_logger()

########################################################
# data preparing
########################################################
def build_loaders(cfg):
    if cfg.dataset in ['cifar10', 'cifar100']:
        c_in = 3
        if cfg.dataset == 'cifar10':
            dataloader = datasets.CIFAR10
            num_classes = 10
            normalization_mean = (0.4914, 0.4822, 0.4465)
            normalization_std = (0.2023, 0.1994, 0.2010)
        else:
            dataloader = datasets.CIFAR100
            num_classes = 100
            normalization_mean = (0.5071, 0.4867, 0.4408)
            normalization_std = (0.2675, 0.2565, 0.2761)

        transform_train = transforms.Compose([
            transforms.RandomCrop(32, padding=4),
            Cutout(),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize(normalization_mean, normalization_std),
        ])

        transform_test = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(normalization_mean, normalization_std),
        ])

        trainset = dataloader(root=cfg.data_dir, train=True, download=True, transform=transform_train)
        testset  = dataloader(root=cfg.data_dir, train=False, download=True, transform=transform_test)

        train_loader = data.DataLoader(trainset, batch_size=cfg.b, shuffle=True,
                                       num_workers=cfg.j)
        test_loader  = data.DataLoader(testset, batch_size=cfg.b, shuffle=False,
                                       num_workers=cfg.j)
        return train_loader, test_loader, c_in, num_classes

    elif cfg.dataset == 'DVSCIFAR10':
        from utils.augmentation import ToPILImage, Resize, Padding, RandomCrop, ToTensor, Normalize, RandomHorizontalFlip
        
        transform_train = transforms.Compose([
        ToPILImage(),
        Resize(48),
        Padding(4),
        RandomCrop(size=48, consistent=True),
        ToTensor(),
        Normalize((0.2728, 0.1295), (0.2225, 0.1290)),
        ])

        transform_test = transforms.Compose([
            ToPILImage(),
            Resize(48),
            ToTensor(),
            Normalize((0.2728, 0.1295), (0.2225, 0.1290)),
        ])
        
        c_in, num_classes = 2, 10
        #tfm = transforms.Compose([ToPILImage(), Resize(48), ToTensor()])
        trainset = CIFAR10DVS(cfg.data_dir, train=True,  use_frame=True, frames_num=cfg.T, split_by='number', normalization=None, transform=transform_train)
        testset  = CIFAR10DVS(cfg.data_dir, train=False, use_frame=True, frames_num=cfg.T, split_by='number', normalization=None, transform=transform_test)

        train_loader = data.DataLoader(trainset, batch_size=cfg.b, shuffle=True,
                                       num_workers=cfg.j)
        test_loader  = data.DataLoader(testset, batch_size=cfg.b, shuffle=False,
                                       num_workers=cfg.j)
        return train_loader, test_loader, c_in, num_classes

    elif cfg.dataset == 'dvsgesture':
        c_in, num_classes = 2, 11
        trainset = DVS128Gesture(root=cfg.data_dir, train=True,  data_type='frame', frames_number=cfg.T, split_by='number')
        testset  = DVS128Gesture(root=cfg.data_dir, train=False, data_type='frame', frames_number=cfg.T, split_by='number')

        train_loader = data.DataLoader(trainset, batch_size=cfg.b, shuffle=True,
                                       num_workers=cfg.j, drop_last=True, pin_memory=True)
        test_loader  = data.DataLoader(testset, batch_size=cfg.b, shuffle=False,
                                       num_workers=cfg.j, drop_last=False, pin_memory=True)
        return train_loader, test_loader, c_in, num_classes

    elif cfg.dataset == 'imagenet':
        num_classes = 1000
        c_in = 3
        traindir = os.path.join(cfg.data_dir, 'train')
        valdir  = os.path.join(cfg.data_dir, 'val')
        normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                         std=[0.229, 0.224, 0.225])

        train_loader = torch.utils.data.DataLoader(
            datasets.ImageFolder(traindir, transforms.Compose([
                transforms.RandomResizedCrop(224),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),
                normalize,
            ])),
            batch_size=cfg.b, shuffle=True, num_workers=cfg.j, pin_memory=True)

        test_loader = torch.utils.data.DataLoader(
            datasets.ImageFolder(valdir, transforms.Compose([
                transforms.Resize(256),
                transforms.CenterCrop(224),
                transforms.ToTensor(),
                normalize,
            ])),
            batch_size=cfg.b, shuffle=False, num_workers=cfg.j, pin_memory=True)

        return train_loader, test_loader, c_in, num_classes
    else:
        raise NotImplementedError(cfg.dataset)

train_loader, test_loader, c_in, num_classes = build_loaders(Cfg)

##########################################################
# model preparing
##########################################################
if Cfg.surrogate == 'sigmoid':
    surrogate_function = surrogate_sj.Sigmoid()
elif Cfg.surrogate == 'rectangle':
    surrogate_function = surrogate_self.Rectangle()
elif Cfg.surrogate == 'triangle':
    surrogate_function = surrogate_sj.PiecewiseQuadratic()
else:
    raise NotImplementedError(Cfg.surrogate)

neuron_model = neuron.Learnable_Threshold_Through_Time_SLTTNeuron

if Cfg.dataset in ['cifar10', 'cifar100']:
    net = spiking_vgg_bn.__dict__[Cfg.model](
        neuron=neuron_model, num_classes=num_classes, neuron_dropout=Cfg.drop_rate,
        tau=Cfg.tau, surrogate_function=surrogate_function, c_in=c_in,
        fc_hw=1, BN=Cfg.BN, T=Cfg.T, v_threshold=0.5
    )
elif Cfg.dataset == 'imagenet':
    net = spiking_resnet_imagenet.__dict__[Cfg.model](
        neuron=neuron_model, num_classes=num_classes, neuron_dropout=Cfg.drop_rate,
        tau=Cfg.tau, surrogate_function=surrogate_function, c_in=3
    )
elif Cfg.dataset in ['DVSCIFAR10','dvsgesture']:
    net = spiking_vgg_bn.__dict__[Cfg.model](
        neuron=neuron_model, num_classes=num_classes, neuron_dropout=Cfg.drop_rate,
        tau=Cfg.tau, surrogate_function=surrogate_function, c_in=c_in,
        fc_hw=1, BN=Cfg.BN, T=Cfg.T, v_threshold=0.5
    )
else:
    raise NotImplementedError(Cfg.dataset)

print('Using model:', Cfg.model)
print('Total Parameters: %.2fM' % (sum(p.numel() for p in net.parameters()) / 1e6))
net.to(device)

thr_params, base_params = [], []
for name, p in net.named_parameters():
    if not p.requires_grad:
        continue
    (thr_params if 'vth_per_t' in name else base_params).append(p)
    print(name)

print(f"threshold params: {len(thr_params)}, base params: {len(base_params)}")

##########################################################
# optimizer preparing
##########################################################
if Cfg.opt == 'SGD':
    optimizer = torch.optim.SGD(
        [{"params": base_params},
         {"params": thr_params, "lr": 0.00078125}],
        lr=Cfg.lr, momentum=Cfg.momentum, weight_decay=Cfg.weight_decay
    )
elif Cfg.opt == 'AdamW':
    optimizer = torch.optim.AdamW(net.parameters(), lr=Cfg.lr, weight_decay=Cfg.weight_decay)
else:
    raise NotImplementedError(Cfg.opt)

if Cfg.lr_scheduler == 'StepLR':
    lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=Cfg.step_size, gamma=Cfg.gamma)
elif Cfg.lr_scheduler == 'CosALR':
    lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=Cfg.T_max)
else:
    raise NotImplementedError(Cfg.lr_scheduler)

scaler = None
if Cfg.amp:
    scaler = amp.GradScaler()

##########################################################
# loading models from checkpoint
##########################################################
start_epoch = 0
max_test_acc = 0.0
if Cfg.resume:
    print('Resuming from', Cfg.resume)
    ckpt = torch.load(Cfg.resume, map_location='cpu')
    net.load_state_dict(ckpt['net'])
    optimizer.load_state_dict(ckpt['optimizer'])
    lr_scheduler.load_state_dict(ckpt['lr_scheduler'])
    start_epoch = ckpt['epoch'] + 1
    max_test_acc = ckpt.get('max_test_acc', 0.0)
    print('start epoch:', start_epoch, ', max test acc:', max_test_acc)

if Cfg.pre_train:
    print('Loading pre-trained from', Cfg.pre_train)
    ckpt = torch.load(Cfg.pre_train, map_location='cpu')
    state_dict2 = collections.OrderedDict([(k, v) for k, v in ckpt['net'].items()])
    net.load_state_dict(state_dict2)
    print('use pre-trained model, max test acc:', ckpt.get('max_test_acc', 0.0))

##########################################################
# output setting
##########################################################
out_dir = os.path.join(
    Cfg.out_dir,
    f"SLTT_{Cfg.dataset}_{Cfg.model}_{Cfg.name}_T{Cfg.T}_tau{Cfg.tau}_e{Cfg.epochs}_bs{Cfg.b}_{Cfg.opt}"
    f"_lr{Cfg.lr}_wd{Cfg.weight_decay}_SG_{Cfg.surrogate}_drop{Cfg.drop_rate}_losslamb{Cfg.loss_lambda}_"
    + ('CosALR_' + str(Cfg.T_max) if Cfg.lr_scheduler=='CosALR' else f"StepLR_{Cfg.step_size}_{Cfg.gamma}")
    + ('_amp' if Cfg.amp else '')
)
os.makedirs(out_dir, exist_ok=True)
print('Output dir:', out_dir)

with open(os.path.join(out_dir, 'args.txt'), 'w', encoding='utf-8') as f:
    f.write(str(Cfg.__dict__))

if Cfg.save_init:
    torch.save({'net': net.state_dict(), 'epoch': 0, 'max_test_acc': 0.0},
               os.path.join(out_dir, 'checkpoint_0.pth'))

writer = SummaryWriter(os.path.join(out_dir, 'logs'), purge_step=start_epoch)

##########################################################
# training and testing
##########################################################
criterion_mse = nn.MSELoss()

# -------------------------------
# Energy monitor using py3nvml
# -------------------------------
nvmlInit()
handle = nvmlDeviceGetHandleByIndex(0)
power_samples = []
sampling = True

def power_sampler(interval=0.2):
    global power_samples, sampling
    while sampling:
        power = nvmlDeviceGetPowerUsage(handle) / 1000  # mW -> W
        power_samples.append(power)
        time.sleep(interval)

def train_one_epoch(epoch, cfg):
    global power_samples, sampling 
    
    power_samples = []
    sampling = True
    th = threading.Thread(target=power_sampler)
    th.start()
    
    time_start = time.time()
    
    net.train()
    batch_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter(); top5 = AverageMeter()

    train_loss_sum = 0.0
    train_acc_sum = 0.0
    train_samples = 0
    
    log_gap = 20

    start = time.time()
    pbar = tqdm(enumerate(train_loader), total=len(train_loader), mininterval=2.0, desc=f"Train[{epoch}]")

    for batch_idx, (frame, label) in pbar:
        if cfg.dataset != 'DVSCIFAR10':
            frame = frame.float().to(device, non_blocking=True)
            if cfg.dataset == 'dvsgesture':
                frame = frame.transpose(0,1)  # T, B, C, H, W
        label = label.to(device, non_blocking=True)
        t_step = cfg.T

        batch_loss_accum = 0.0

        if not cfg.online_update:
            optimizer.zero_grad(set_to_none=True)

        for t in range(t_step):
            if cfg.online_update:
                optimizer.zero_grad(set_to_none=True)

            if cfg.dataset == 'DVSCIFAR10':
                input_frame = frame[t].float().to(device, non_blocking=True)
            elif cfg.dataset == 'dvsgesture':
                input_frame = frame[t]
            else:
                input_frame = frame
            
            if cfg.amp:
                with amp.autocast():
                    if t == 0:
                        out_fr = net(input_frame, t=t)
                        total_fr = out_fr.clone().detach()
                    else:
                        out_fr = net(input_frame, t=t)
                        total_fr += out_fr.clone().detach()
                    if cfg.loss_lambda > 0.0:
                        if cfg.mse_n_reg:
                            label_one_hot = F.one_hot(label, num_classes).float()
                        else:
                            label_one_hot = torch.zeros_like(out_fr).fill_(cfg.loss_means).to(out_fr.device)
                        mse_loss = criterion_mse(out_fr, label_one_hot)
                        loss = ((1 - cfg.loss_lambda) * F.cross_entropy(out_fr, label) + cfg.loss_lambda * mse_loss) / t_step
                    else:
                        loss = F.cross_entropy(out_fr, label) / t_step

                scaler.scale(loss).backward()
                if cfg.online_update:
                    scaler.step(optimizer); scaler.update()
                    
            else:
                if t == 0:
                    out_fr = net(input_frame, t=t)
                    total_fr = out_fr.clone().detach()
                else:
                    out_fr = net(input_frame, t=t)
                    total_fr += out_fr.clone().detach()
                if cfg.loss_lambda > 0.0:
                    label_one_hot = torch.zeros_like(out_fr).fill_(cfg.loss_means).to(out_fr.device)
                    if cfg.mse_n_reg:
                        label_one_hot = F.one_hot(label, num_classes).float()
                    mse_loss = criterion_mse(out_fr, label_one_hot)
                    loss = ((1 - cfg.loss_lambda) * F.cross_entropy(out_fr, label) + cfg.loss_lambda * mse_loss) / t_step
                else:
                    loss = F.cross_entropy(out_fr, label) / t_step

                loss.backward()
                if cfg.online_update:
                    #for name, p in net.named_parameters():
                    #    if "vth_per_t" in name:
                    #        if p.grad is None:
                    #            print(f"[NO GRAD] {name}")
                    #        else:
                    #            print(f"[GRAD] {name}: grad_mean={p.grad.abs().mean().item():.6e}")
                    optimizer.step()

            batch_loss_accum += float(loss.item())
            train_loss_sum += loss.item() * label.numel()

        if not cfg.online_update:
            if cfg.amp:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
        
        #for name, param in net.named_parameters():
        #    if "vth_per_t" in name:
        #        print(f"{name}: shape={param.shape}, mean={param.data.mean().item():.4f}, values={param.data}")
        
        prec1, prec5 = accuracy(total_fr.data, label.data, topk=(1,5))
        losses.update(batch_loss_accum, input_frame.size(0))
        top1.update(prec1.item(), input_frame.size(0))
        top5.update(prec5.item(), input_frame.size(0))

        train_samples += label.numel()
        train_acc_sum += (total_fr.argmax(1) == label).float().sum().item()

        functional.reset_net(net)

        batch_time.update(time.time() - start)
        start = time.time()
        
        if batch_idx % log_gap == 0 or batch_idx == len(train_loader):
            pbar.set_postfix(loss=f"{losses.avg:.4f}", top1=f"{top1.avg:.4f}", top5=f"{top5.avg:.4f}")

    time_end = time.time()
    epoch_latency = time_end - time_start
    
    sampling = False
    th.join()
    
    if len(power_samples) > 0:
        avg_power = sum(power_samples) / len(power_samples)
        energy = avg_power * epoch_latency
    else:
        avg_power = 0.0
        energy = 0.0
    
    print("One training epoch latency: {:.4}s | Avg Power: {:.4}W | Energy: {:.4f}J".format(
        epoch_latency, avg_power, energy))
    
    train_loss = train_loss_sum / max(1, train_samples)
    train_acc  = train_acc_sum / max(1, train_samples)
    writer.add_scalar('train_loss', train_loss, epoch)
    writer.add_scalar('train_acc',  train_acc,  epoch)
    return train_loss, train_acc, epoch_latency, avg_power, energy

@torch.no_grad()
def validate(epoch, cfg):
    net.eval()
    losses = AverageMeter()
    top1 = AverageMeter(); top5 = AverageMeter()
    
    log_gap = 20

    test_loss_sum = 0.0
    test_acc_sum  = 0.0
    test_samples  = 0

    pbar = tqdm(enumerate(test_loader), total=len(test_loader), mininterval=2.0, desc=f"Test [{epoch}]")

    for batch_idx, (frame, label) in pbar:
        if cfg.dataset != 'DVSCIFAR10':
            frame = frame.float().to(device, non_blocking=True)
            if cfg.dataset == 'dvsgesture':
                frame = frame.transpose(0,1)
        label = label.to(device, non_blocking=True)
        t_step = cfg.T

        total_loss = 0.0

        for t in range(t_step):
            if cfg.dataset == 'DVSCIFAR10':
                input_frame = frame[t].float().to(device, non_blocking=True)
            elif cfg.dataset == 'dvsgesture':
                input_frame = frame[t]
            else:
                input_frame = frame

            out_fr = net(input_frame, t=t)
            if t == 0:
                total_fr = out_fr.detach().clone()
            else:
                total_fr += out_fr.detach().clone()

            if cfg.loss_lambda > 0.0:
                if cfg.mse_n_reg:
                    label_one_hot = F.one_hot(label, num_classes).float()
                else:
                    label_one_hot = torch.zeros_like(out_fr).fill_(cfg.loss_means).to(out_fr.device)
                mse_loss = criterion_mse(out_fr, label_one_hot)
                loss = ((1 - cfg.loss_lambda) * F.cross_entropy(out_fr, label) + cfg.loss_lambda * mse_loss) / t_step
            else:
                loss = F.cross_entropy(out_fr, label) / t_step
            total_loss += float(loss.item())

        test_samples += label.numel()
        test_loss_sum += total_loss * label.numel()
        test_acc_sum  += (total_fr.argmax(1) == label).float().sum().item()

        functional.reset_net(net)

        prec1, prec5 = accuracy(total_fr.data, label.data, topk=(1,5))
        losses.update(total_loss, n=input_frame.size(0))
        top1.update(prec1.item(), n=input_frame.size(0))
        top5.update(prec5.item(), n=input_frame.size(0))
        
        if batch_idx % log_gap == 0 or batch_idx == len(test_loader):
            pbar.set_postfix(loss=f"{losses.avg:.4f}", top1=f"{top1.avg:.4f}", top5=f"{top5.avg:.4f}")

    test_loss = test_loss_sum / max(1, test_samples)
    test_acc  = test_acc_sum  / max(1, test_samples)
    writer.add_scalar('test_loss', test_loss, epoch)
    writer.add_scalar('test_acc',  test_acc,  epoch)
    return test_loss, test_acc


def run_training(cfg, start_epoch=0, max_test_acc=0.0):
    best = max_test_acc
    for epoch in range(start_epoch, cfg.epochs):
        epoch_t0 = time.time()

        train_loss, train_acc, epoch_latency, avg_power, energy = train_one_epoch(epoch, cfg)
        if cfg.lr_scheduler is not None:
            lr_scheduler.step()

        test_loss, test_acc = validate(epoch, cfg)

        save_max = test_acc > best
        best = max(best, test_acc)
        ckpt = {
            'net': net.state_dict(),
            'optimizer': optimizer.state_dict(),
            'lr_scheduler': lr_scheduler.state_dict(),
            'epoch': epoch,
            'max_test_acc': best
        }
        torch.save(ckpt, os.path.join(out_dir, 'checkpoint_latest.pth'))
        if save_max:
            torch.save(ckpt, os.path.join(out_dir, 'checkpoint_max.pth'))

        total_time = time.time() - epoch_t0
        eta_str = (datetime.datetime.now() + datetime.timedelta(seconds=total_time * (cfg.epochs - epoch - 1))).strftime("%Y-%m-%d %H:%M:%S")
        
        _append_txt_log(
            txt_log_path,
            epoch=epoch,
            train_loss=train_loss,
            train_acc=train_acc,
            test_loss=test_loss,
            test_acc=test_acc,
            max_test_acc=best,
            epoch_latency=epoch_latency,
            avg_power=avg_power,
            energy=energy,
            total_time=total_time
            )
        
        print(f'epoch={epoch}, train_loss={train_loss:.6f}, train_acc={train_acc:.6f}, '
              f'test_loss={test_loss:.6f}, test_acc={test_acc:.6f}, max_test_acc={best:.6f}, '
              f'total_time={total_time:.2f}s, est_finish={eta_str}')

        if torch.cuda.is_available():
            try:
                mem_gb = torch.cuda.max_memory_reserved(0) / 1024 / 1024 / 1024
            except:
                mem_gb = torch.cuda.max_memory_allocated(0) / 1024 / 1024 / 1024
            print(f"after one epoch: {mem_gb:.2f} GB")

    return best

best_acc = run_training(Cfg, start_epoch=start_epoch, max_test_acc=max_test_acc)
print('Training done. Best Acc =', best_acc)

Running on: cuda
Using model: spiking_vgg11_lttt_sw
Total Parameters: 9.27M
layer1.0.conv.weight
layer1.0.conv.bias
layer1.0.conv.gain
layer1.0.neuron.vth_per_t
layer2.0.conv.weight
layer2.0.conv.bias
layer2.0.conv.gain
layer2.0.neuron.vth_per_t
layer3.0.conv.weight
layer3.0.conv.bias
layer3.0.conv.gain
layer3.0.neuron.vth_per_t
layer3.1.conv.weight
layer3.1.conv.bias
layer3.1.conv.gain
layer3.1.neuron.vth_per_t
layer4.0.conv.weight
layer4.0.conv.bias
layer4.0.conv.gain
layer4.0.neuron.vth_per_t
layer4.1.conv.weight
layer4.1.conv.bias
layer4.1.conv.gain
layer4.1.neuron.vth_per_t
layer5.0.conv.weight
layer5.0.conv.bias
layer5.0.conv.gain
layer5.0.neuron.vth_per_t
layer5.1.conv.weight
layer5.1.conv.bias
layer5.1.conv.gain
layer5.1.neuron.vth_per_t
classifier.1.weight
classifier.1.bias
threshold params: 8, base params: 26
Output dir: ./logs/SLTT_cifar100_spiking_vgg11_lttt_sw__T6_tau1.1_e100_bs1_SGD_lr0.00078125_wd0.0_SG_triangle_drop0.0_losslamb0.1_CosALR_300


Train[0]: 100%|██████████| 50000/50000 [23:32<00:00, 35.41it/s, loss=3.6209, top1=9.3876, top5=29.8513]


One training epoch latency: 1.412e+03s | Avg Power: 172.1W | Energy: 243036.1835J


Test [0]: 100%|██████████| 10000/10000 [01:40<00:00, 99.53it/s, loss=3.2726, top1=17.0825, top5=43.4526]


epoch=0, train_loss=3.620877, train_acc=0.093880, test_loss=3.272507, test_acc=0.170800, max_test_acc=0.170800, total_time=1512.68s, est_finish=2025-09-05 04:58:56
after one epoch: 0.25 GB


...

epoch=98, train_loss=0.390049, train_acc=0.957300, test_loss=1.330418, test_acc=0.697700, max_test_acc=0.697700, total_time=1483.63s, est_finish=2025-09-05 04:12:17
after one epoch: 0.25 GB


Train[99]: 100%|██████████| 50000/50000 [23:05<00:00, 36.09it/s, loss=0.3864, top1=95.9144, top5=99.5378]


One training epoch latency: 1.386e+03s | Avg Power: 172.8W | Energy: 239379.6507J


Test [99]: 100%|██████████| 10000/10000 [01:36<00:00, 103.26it/s, loss=1.3410, top1=69.8427, top5=89.9008]


epoch=99, train_loss=0.386433, train_acc=0.959160, test_loss=1.340408, test_acc=0.698400, max_test_acc=0.698400, total_time=1482.73s, est_finish=2025-09-05 04:12:16
after one epoch: 0.25 GB
Training done. Best Acc = 0.6984
